# 03 — Train DAS on Controlled NLI

We trained a vanilla activation-patching baseline in notebook 02. Now we
lift it to **Distributed Alignment Search**: at a *fixed* layer / component /
token position, we learn an orthogonal rotation so that the rotated
low-rank subspace causally implements the high-level variable
`lexical_relation`.

Pipeline:

1. Build train + held-out counterfactual datasets targeting
   `lexical_relation`.
2. Pick one promising `(layer, component, position)` site (here we
   hard-code one informed by the heatmap from notebook 02; in a real
   project you'd peak-pick programmatically from the saved CSV).
3. Train a `LowRankRotatedSpaceIntervention` (`intervention_dim = 4`,
   matching the 4-valued lexical-relation variable) on the train set.
4. Evaluate factual accuracy + IIA on the held-out set.
5. Compare to two controls:
    - **Random-source control.** Replace each source with a randomly
      shuffled source from the eval set. IIA should collapse to chance.
    - **Wrong-variable control.** Re-evaluate the *same trained* DAS
      against counterfactual labels computed for a *different* high-level
      variable (`premise_word_identity`). IIA should also drop, because
      the subspace was aligned to `lexical_relation` specifically.

Runs on CPU in a few minutes with `distilgpt2`.

## 1. Setup

In [ ]:
import os, sys, json, copy, random, pathlib

_HERE = pathlib.Path.cwd()
for _p in [_HERE, *_HERE.parents]:
    if (_p / 'src').is_dir():
        PROJECT_ROOT = _p
        break
else:
    raise RuntimeError('Could not locate the project root (no src/ found).')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import set_seed
from src.utils.seed import get_device
from src.models import load_causal_lm
from src.data import build_counterfactual_dataset, ID2LABEL
from src.metrics import LabelVerbalizer
from src.interventions import (
    make_das_config,
    train_das_alignment,
    evaluate_das_iia,
)

set_seed(0)

MODEL_NAME = 'distilgpt2'
DEVICE     = get_device()

# Site informed by the patching heatmap in notebook 02. For distilgpt2
# (6 layers) the residual stream around layer 4 at the hypothesis-word
# position tends to carry the relation signal. Adjust after viewing your
# own heatmap.
LAYER            = 4
COMPONENT        = 'block_output'
INTERVENTION_DIM = 4   # matches the 4 lexical relations
TARGET_VAR       = 'lexical_relation'

# Data sizing.
N_TRAIN   = 64
N_EVAL    = 32
BATCH_SZ  = 8
NUM_EPOCHS = 4
LR         = 1e-3

OUT_DIR = PROJECT_ROOT / 'outputs'
(OUT_DIR / 'das').mkdir(parents=True, exist_ok=True)
print('DEVICE =', DEVICE, ' | site =', (LAYER, COMPONENT, f'dim={INTERVENTION_DIM}'))

## 2. Load model + verbalizer

In [ ]:
tokenizer, model = load_causal_lm(MODEL_NAME, device=DEVICE)
n_layers = getattr(model.config, 'n_layer', getattr(model.config, 'num_hidden_layers', None))
print(f'Loaded {MODEL_NAME} on {DEVICE} (n_layers={n_layers}, hidden={model.config.n_embd})')

verbalizer = LabelVerbalizer.from_tokenizer(
    tokenizer,
    {'entailment': ' yes', 'neutral': ' maybe', 'contradiction': ' no'},
)
verbalizer.token_ids

## 3. Build counterfactual datasets

Both targeting the same high-level variable `lexical_relation`, but with
different RNG seeds so the train/eval splits don't share examples.
We require `cf_label != base_label` so chance-level IIA is meaningfully
below 1.

In [ ]:
train_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    n_examples=N_TRAIN, seed=0, require_label_change=True,
)
eval_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    n_examples=N_EVAL, seed=42, require_label_change=True,
)
print(f'train: {len(train_ds)}   eval: {len(eval_ds)}')

# Sanity peek: how does the *unintervened* base model do on these prompts?
from src.metrics.logits import decode_label
with torch.no_grad():
    out = model(
        input_ids=eval_ds._base_input_ids.to(DEVICE),
        attention_mask=eval_ds._base_attn.to(DEVICE),
    )
    preds = decode_label(out.logits, verbalizer, attention_mask=eval_ds._base_attn.to(DEVICE)).cpu()
gold = torch.tensor([ex.base_label_id for ex in eval_ds.examples])
print(f'unintervened base accuracy on eval set: {(preds == gold).float().mean().item():.2f}')

## 4. Configure DAS and train

`make_das_config` builds an `IntervenableConfig` whose intervention is a
`LowRankRotatedSpaceIntervention` of dimension `INTERVENTION_DIM` at the
chosen `(layer, component)`. `train_das_alignment` freezes the base LM
and trains only the rotation matrix.

In [ ]:
das_config = make_das_config(
    model,
    layer=LAYER,
    component=COMPONENT,
    intervention_dim=INTERVENTION_DIM,
    unit='pos',
)
print(das_config.__dict__['_das_meta'])

log_path = OUT_DIR / 'das' / f'train_history_{TARGET_VAR}_L{LAYER}_{COMPONENT}.json'
out = train_das_alignment(
    model, tokenizer, train_ds, das_config,
    num_epochs=NUM_EPOCHS, lr=LR, batch_size=BATCH_SZ,
    device=DEVICE, verbalizer=verbalizer,
    log_path=str(log_path), progress=True,
)
intervenable_lex = out.intervenable
history = out.history
print('saved history to', log_path)
print('final train IIA:', history[-1]['train_iia'])

In [ ]:
# Plot training curve.
hist_df = pd.DataFrame(history)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(hist_df['epoch'], hist_df['loss'], marker='o', label='train loss', color='tab:red')
ax1.set_xlabel('epoch'); ax1.set_ylabel('CE loss', color='tab:red')
ax2 = ax1.twinx()
ax2.plot(hist_df['epoch'], hist_df['train_iia'], marker='s', label='train IIA', color='tab:blue')
ax2.set_ylabel('train IIA', color='tab:blue'); ax2.set_ylim(0, 1.0)
ax1.set_title(f'DAS training curve (L{LAYER} {COMPONENT}, d={INTERVENTION_DIM})')
fig.tight_layout()
fig.savefig(OUT_DIR / 'das' / 'training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Evaluate on held-out CF pairs (correct alignment)

In [ ]:
res_lex = evaluate_das_iia(intervenable_lex, eval_ds, tokenizer, device=DEVICE, verbalizer=verbalizer)
print(f"factual_accuracy = {res_lex['factual_accuracy']:.3f}")
print(f"IIA              = {res_lex['iia']:.3f}   (n={res_lex['n_examples']})")
print('IIA per class    :', {k: round(v, 3) for k, v in res_lex['iia_per_class'].items()})
res_lex['confusion']

## 6. Random-source control

We **shuffle** the source rows of the eval dataset so each base is paired
with a random other source. The high-level counterfactual labels are
unchanged, so if the patched activation no longer actually carries the
correct value of `lexical_relation`, IIA should fall to chance.

In [ ]:
def shuffle_sources(dataset, seed=0):
    """Return a shallow-copied dataset whose source rows are permuted.
    cf labels and intervention positions stay tied to the *base*."""
    rng = np.random.RandomState(seed)
    n = len(dataset)
    perm = rng.permutation(n)
    # Re-rolling all-same-position pathological perms is overkill for
    # n>=4; just make sure no example is mapped to itself.
    for _ in range(10):
        if not np.any(perm == np.arange(n)): break
        perm = rng.permutation(n)
    new_ds = copy.copy(dataset)
    new_ds._source_input_ids = dataset._source_input_ids[perm].clone()
    new_ds._source_attn      = dataset._source_attn[perm].clone()
    new_ds.examples = list(dataset.examples)  # cf labels untouched
    return new_ds

eval_random = shuffle_sources(eval_ds, seed=1)
res_random = evaluate_das_iia(intervenable_lex, eval_random, tokenizer, device=DEVICE, verbalizer=verbalizer)
print(f"random-source IIA = {res_random['iia']:.3f}   (n={res_random['n_examples']})")
res_random['confusion']

## 7. Wrong-variable control

We build a second eval set whose counterfactual labels are computed for
the **wrong** intermediate variable (`premise_word_identity`) at the
**same intervention site** (hypothesis-word position) as the trained
DAS. The trained rotation should not generalise: IIA should drop.

In [ ]:
eval_wrong = build_counterfactual_dataset(
    tokenizer,
    target_variable='premise_word_identity',
    intervention_word='hypothesis_word',   # keep patch site fixed
    n_examples=N_EVAL, seed=42, require_label_change=True,
)
print(f'wrong-variable eval set: {len(eval_wrong)}')
res_wrong = evaluate_das_iia(intervenable_lex, eval_wrong, tokenizer, device=DEVICE, verbalizer=verbalizer)
print(f"wrong-variable IIA = {res_wrong['iia']:.3f}")
res_wrong['confusion']

## 8. Summary

In [ ]:
summary = pd.DataFrame([
    {'condition': 'lexical_relation (trained variable)',
     'factual_acc': res_lex['factual_accuracy'],
     'IIA':         res_lex['iia'],
     'n':           res_lex['n_examples']},
    {'condition': 'random-source control',
     'factual_acc': res_random['factual_accuracy'],
     'IIA':         res_random['iia'],
     'n':           res_random['n_examples']},
    {'condition': 'wrong-variable control (premise_word_identity)',
     'factual_acc': res_wrong['factual_accuracy'],
     'IIA':         res_wrong['iia'],
     'n':           res_wrong['n_examples']},
])
summary['IIA'] = summary['IIA'].round(3)
summary['factual_acc'] = summary['factual_acc'].round(3)
summary.to_csv(OUT_DIR / 'das' / 'iia_summary.csv', index=False)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(summary['condition'], summary['IIA'], color=['tab:blue', 'tab:gray', 'tab:orange'])
ax.axhline(1/3, color='black', linestyle='--', linewidth=1, label='3-way chance')
ax.set_ylim(0, 1.05); ax.set_ylabel('IIA')
ax.set_title(f'DAS IIA at L{LAYER} {COMPONENT} (d={INTERVENTION_DIM})')
for tick in ax.get_xticklabels():
    tick.set_rotation(15); tick.set_ha('right')
ax.legend(loc='upper right')
fig.tight_layout()
fig.savefig(OUT_DIR / 'das' / 'iia_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation cheat sheet

- **`lexical_relation` IIA**  — ceiling we can reach with this site /
  subspace dimension. If this is near chance, the alignment is failing;
  try a different layer, a different component, or a larger
  `INTERVENTION_DIM`.
- **random-source IIA**  — lower bound. Should be at or near 3-way
  chance (~0.33). If it's high, your dataset is leaking the answer
  somewhere outside the patched site.
- **wrong-variable IIA**  — specificity check. Should also be near
  chance. If it's close to the `lexical_relation` IIA, the trained
  subspace is encoding something more general (e.g. "the hypothesis
  word identity") and isn't specifically aligned to `lexical_relation`.

### Next steps

- Sweep `(LAYER, COMPONENT, INTERVENTION_DIM)` and pick the site that
  maximises the IIA *gap* between the trained and the wrong-variable
  conditions.
- Swap the LowRankRotatedSpaceIntervention for the boundary-learning
  Boundless DAS variant (see `Boundless_DAS.ipynb`).
- Move from `distilgpt2` to `gpt2` / `gpt2-medium` and see how the
  picture changes.